# Greedy Best-Fit Decreasing

This notebook integrates a Greedy Best-Fit Decreasing packing strategy into the project and uses the shared CSV dataset in `../data/data.csv`.

In [ ]:
import csv
import math
import random
from itertools import permutations
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data" / "data.csv"

def load_random_boxes(csv_path=DATA_PATH, n=100, seed=None):
    if seed is not None:
        random.seed(seed)

    with open(csv_path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    selected_rows = random.sample(rows, min(n, len(rows)))
    boxes = []

    for row in selected_rows:
        boxes.append({
            "id": row["id"],
            "name": row["name"],
            "l": int(row["length"]),
            "w": int(row["width"]),
            "h": int(row["height"]),
            "weight": float(row["weight"]),
            "fragile": row["fragile"].strip().lower() == "true",
        })

    return boxes

In [ ]:
CONTAINER_L = 590
CONTAINER_W = 235
CONTAINER_H = 239

class Container:
    def __init__(self, length=CONTAINER_L, width=CONTAINER_W, height=CONTAINER_H):
        self.length = length
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        self.placed_boxes = []
        self.unplaced_boxes = []
        self.free_spaces = [{
            "x": 0,
            "y": 0,
            "z": 0,
            "L": self.length,
            "W": self.width,
            "H": self.height,
        }]

    def get_orientations(self, box):
        l, w, h = box["l"], box["w"], box["h"]

        if box["fragile"]:
            return list({(l, w, h), (w, l, h)})

        return list({
            (l, w, h), (l, h, w),
            (w, l, h), (w, h, l),
            (h, l, w), (h, w, l),
        })

    def fits(self, l, w, h, space):
        return l <= space["L"] and w <= space["W"] and h <= space["H"]

    def split_space(self, space, l, w, h, x, y, z):
        new_spaces = []

        if space["L"] > l:
            new_spaces.append({
                "x": x + l,
                "y": y,
                "z": z,
                "L": space["L"] - l,
                "W": space["W"],
                "H": space["H"],
            })

        if space["W"] > w:
            new_spaces.append({
                "x": x,
                "y": y + w,
                "z": z,
                "L": space["L"],
                "W": space["W"] - w,
                "H": space["H"],
            })

        if space["H"] > h:
            new_spaces.append({
                "x": x,
                "y": y,
                "z": z + h,
                "L": space["L"],
                "W": space["W"],
                "H": space["H"] - h,
            })

        return new_spaces

    def place_box(self, box, choice):
        if choice is None:
            self.unplaced_boxes.append({
                "id": box["id"],
                "name": box["name"],
                "reason": "no valid placement found",
            })
            return False

        space = choice["space"]
        l, w, h = choice["dims"]
        x, y, z = space["x"], space["y"], space["z"]

        placed = {
            "id": box["id"],
            "name": box["name"],
            "weight": box["weight"],
            "x": x,
            "y": y,
            "z": z,
            "l": l,
            "w": w,
            "h": h,
        }

        self.placed_boxes.append(placed)
        self.free_spaces.remove(space)
        self.free_spaces.extend(self.split_space(space, l, w, h, x, y, z))
        return True

    def evaluate_stability(self):
        if not self.placed_boxes:
            return {"weight_score": 0.0, "support_score": 0.0, "stability_score": 0.0}

        pairs_correct = 0
        pairs_total = 0

        for i, box_a in enumerate(self.placed_boxes):
            for box_b in self.placed_boxes[i + 1:]:
                w_a = box_a["weight"]
                w_b = box_b["weight"]
                z_a = box_a["z"]
                z_b = box_b["z"]

                if w_a == w_b:
                    pairs_correct += 1
                elif w_a > w_b:
                    if z_a <= z_b:
                        pairs_correct += 1
                else:
                    if z_b <= z_a:
                        pairs_correct += 1

                pairs_total += 1

        weight_score = (pairs_correct / pairs_total * 100) if pairs_total else 100.0

        min_overlap_ratio = 0.30
        supported_count = 0

        for box in self.placed_boxes:
            if box["z"] == 0:
                supported_count += 1
                continue

            box_x1, box_x2 = box["x"], box["x"] + box["l"]
            box_y1, box_y2 = box["y"], box["y"] + box["w"]
            box_base_area = box["l"] * box["w"]
            supported_area = 0.0

            for other in self.placed_boxes:
                if other["id"] == box["id"]:
                    continue

                other_top = other["z"] + other["h"]
                if other_top != box["z"]:
                    continue

                ox1, ox2 = other["x"], other["x"] + other["l"]
                oy1, oy2 = other["y"], other["y"] + other["w"]

                overlap_x = max(0, min(box_x2, ox2) - max(box_x1, ox1))
                overlap_y = max(0, min(box_y2, oy2) - max(box_y1, oy1))
                supported_area += overlap_x * overlap_y

            ratio = supported_area / box_base_area if box_base_area > 0 else 0
            if ratio >= min_overlap_ratio:
                supported_count += 1

        support_score = (supported_count / len(self.placed_boxes)) * 100
        stability_score = (weight_score * 0.4) + (support_score * 0.6)

        return {
            "weight_score": round(weight_score, 2),
            "support_score": round(support_score, 2),
            "stability_score": round(stability_score, 2),
        }

    def pack_boxes(self, boxes):
        self.reset()

        for box in boxes:
            choice = self.get_best_placement(box)
            self.place_box(box, choice)

        packed_volume = sum(box["l"] * box["w"] * box["h"] for box in self.placed_boxes)
        container_volume = self.length * self.width * self.height
        utilization = (packed_volume / container_volume) * 100 if container_volume else 0.0

        stability = self.evaluate_stability()

        return {
            "placed": len(self.placed_boxes),
            "placed_count": len(self.placed_boxes),
            "unplaced": len(self.unplaced_boxes),
            "unplaced_count": len(self.unplaced_boxes),
            "utilization": utilization,
            "stability": stability,
            "placed_boxes": self.placed_boxes,
            "unplaced_boxes": self.unplaced_boxes,
        }

    def get_best_placement(self, box):
        best_choice = None

        for (l, w, h) in self.get_orientations(box):
            for space in self.free_spaces:
                if not self.fits(l, w, h, space):
                    continue

                score = (space["z"], space["x"], space["y"])
                if best_choice is None or score < best_choice["score"]:
                    best_choice = {
                        "space": space,
                        "dims": (l, w, h),
                        "score": score,
                    }

        return best_choice

In [ ]:
class GreedyBFD:
    def __init__(self, container, boxes):
        self.container = container
        self.boxes = boxes

    def _volume(self, box):
        return box["l"] * box["w"] * box["h"]

    def _sort_decreasing(self, boxes):
        return sorted(boxes, key=self._volume, reverse=True)

    def _best_fit_score(self, space, dims):
        l, w, h = dims
        return (space["L"] * space["W"] * space["H"]) - (l * w * h)

    def _find_best_fit_placement(self, box):
        best_choice = None

        for (l, w, h) in self.container.get_orientations(box):
            for space in self.container.free_spaces:
                if not self.container.fits(l, w, h, space):
                    continue

                leftover = self._best_fit_score(space, (l, w, h))
                candidate_key = (leftover, space["z"], space["x"], space["y"] )

                if best_choice is None or candidate_key < best_choice["key"]:
                    best_choice = {
                        "space": space,
                        "dims": (l, w, h),
                        "score": leftover,
                        "key": candidate_key,
                    }

        return best_choice

    def run(self):
        sorted_boxes = self._sort_decreasing(self.boxes)
        self.container.reset()

        for box in sorted_boxes:
            choice = self._find_best_fit_placement(box)
            self.container.place_box(box, choice)

        packed_volume = sum(
            box["l"] * box["w"] * box["h"] for box in self.container.placed_boxes
        )
        container_volume = self.container.length * self.container.width * self.container.height
        utilization = (packed_volume / container_volume) * 100 if container_volume else 0.0
        stability = self.container.evaluate_stability()

        best_result = {
            "placed": len(self.container.placed_boxes),
            "placed_count": len(self.container.placed_boxes),
            "unplaced": len(self.container.unplaced_boxes),
            "unplaced_count": len(self.container.unplaced_boxes),
            "utilization": utilization,
            "stability": stability,
            "placed_boxes": self.container.placed_boxes,
            "unplaced_boxes": self.container.unplaced_boxes,
        }

        score = (0.7 * utilization) + (0.3 * stability["stability_score"])

        return {
            "sorted_boxes": sorted_boxes,
            "best_solution": sorted_boxes,
            "best_score": score,
            "best_result": best_result,
        }

In [ ]:
boxes = load_random_boxes(n=100, seed=42)
container = Container()
bfd = GreedyBFD(container, boxes)
result = bfd.run()

print("========== PACKING RESULTS ==========")
print(f"Placed boxes   : {result['best_result']['placed']}")
print(f"Unplaced boxes : {result['best_result']['unplaced']}")
print()
print(f"Utilization : {result['best_result']['utilization']:.2f}%")
print(f"Stability   : {result['best_result']['stability']}")
print(f"Score       : {result['best_score']:.2f}")
print()
print("Placed boxes:")
for box in result['best_result']['placed_boxes']:
    print(box)

print()
print("Unplaced boxes:")
for box in result['best_result']['unplaced_boxes']:
    print(box)